In [1]:
# ============================================================
# NOTEBOOK A: PREPROCESSING EDGE-IIoTset
# ============================================================

# ── CELL 1: Import ────────────────────────────────────────────
import csv, os, gc, subprocess, textwrap
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

TEMP_FILE = '/kaggle/working/temp_selected.csv'
OUTPUT    = '/kaggle/working/edge_iiotset_clean.csv'
SCRIPT    = '/kaggle/working/pass12.py'
print("Libraries loaded ✓")


# ── CELL 2: Mapping ───────────────────────────────────────────
ATTACK_GROUPS = {
    'DoS_DDoS':              ['ddos','dos','flood','slowloris','rudy','syn'],
    'Information_Gathering': ['recon','scan','fingerprint','discovery',
                              'probe','password','brute','credential'],
    'MITM':                  ['mitm','arp_spoof','dns_spoof','spoofing','arp','dns'],
    'Injection':             ['injection','sql','xss','command','web_attack'],
    'Malware':               ['backdoor','ransomware','uploading','malware',
                              'trojan','worm','virus'],
}

def map_to_group(attack_type):
    s = str(attack_type).lower().strip()
    if any(x in s for x in ['normal','benign','legitimate']):
        return 'Benign'
    for group, keywords in ATTACK_GROUPS.items():
        if any(kw in s for kw in keywords):
            return group
    return f'UNMAPPED_{attack_type}'

print("Mapping ready ✓")


# ── CELL 3: Pass 1 & 2 via Subprocess ────────────────────────
if os.path.exists(TEMP_FILE):
    print(f"✅ Temp file ada ({os.path.getsize(TEMP_FILE)/1024**2:.1f} MB) — skip")
else:
    print("Menjalankan Pass 1 & 2 via subprocess...")
    code = textwrap.dedent("""
        import csv, gc, numpy as np
        from collections import defaultdict
        FILE = ('/kaggle/input/datasets/mohamedamineferrag/'
                'edgeiiotset-cyber-security-dataset-of-iot-iiot/'
                'Edge-IIoTset dataset/Selected dataset for ML and DL/'
                'ML-EdgeIIoT-dataset.csv')
        TEMP = '/kaggle/working/temp_selected.csv'
        SAMPLES = {'Benign':24000,'DoS_DDoS':12000,
                   'Information_Gathering':6000,'MITM':1200,
                   'Injection':6000,'Malware':12000}
        GROUPS = {
            'DoS_DDoS':['ddos','dos','flood','slowloris','rudy','syn'],
            'Information_Gathering':['recon','scan','fingerprint','discovery',
                                     'probe','password','brute','credential'],
            'MITM':['mitm','arp_spoof','dns_spoof','spoofing','arp','dns'],
            'Injection':['injection','sql','xss','command','web_attack'],
            'Malware':['backdoor','ransomware','uploading','malware','trojan','worm'],
        }
        def g(a):
            s=str(a).lower().strip()
            if any(x in s for x in ['normal','benign']): return 'Benign'
            for grp,kws in GROUPS.items():
                if any(k in s for k in kws): return grp
            return 'UNMAPPED'
        np.random.seed(42)
        buckets=defaultdict(list)
        with open(FILE,'r',buffering=65536,encoding='utf-8',errors='replace') as f:
            r=csv.reader(f); h=next(r)
            col=next((i for i,c in enumerate(h) if 'attack' in c.lower() and 'type' in c.lower()),None)
            for i,row in enumerate(r):
                if len(row)>col:
                    grp=g(row[col])
                    if grp!='UNMAPPED': buckets[grp].append(i)
                if (i+1)%500000==0: print(f'{i+1:,}')
        sel=set()
        for grp,n in SAMPLES.items():
            idx=buckets.get(grp,[])
            na=min(n,len(idx))
            sel.update(np.random.choice(idx,na,replace=False).tolist())
            print(f'{grp}:{na}')
        del buckets; gc.collect()
        w=0
        with open(FILE,'r',buffering=65536,encoding='utf-8',errors='replace') as fi, \
             open(TEMP,'w',buffering=65536,encoding='utf-8') as fo:
            fo.write(fi.readline())
            for i,line in enumerate(fi):
                if i in sel: fo.write(line); w+=1
        print(f'Written:{w}')
    """)
    with open(SCRIPT, 'w') as f:
        f.write(code)
    subprocess.run(['python', SCRIPT], text=True)
    print(f"✅ {os.path.getsize(TEMP_FILE)/1024**2:.1f} MB")
    gc.collect()


# ── CELL 4: Baca dengan Polars ───────────────────────────────
print(f"\nMembaca {TEMP_FILE} dengan polars...")
import polars as pl

df = pl.read_csv(
    TEMP_FILE,
    ignore_errors=True,
    infer_schema_length=500,
    null_values=['','nan','NaN','NA','null','inf','-inf'],
    truncate_ragged_lines=True
).to_pandas()

print(f"Shape: {df.shape}")
print(f"RAM  : {df.memory_usage(deep=True).sum()/1024**2:.1f} MB")


# ── CELL 5: Bersihkan ─────────────────────────────────────────
# Step 1: Drop kolom string kecuali Attack_type
# (IP address, timestamp — bukan fitur ML)
string_cols = [c for c in df.columns
               if df[c].dtype == 'object' and c != 'Attack_type']
print(f"\nDrop {len(string_cols)} kolom string: {string_cols}")
df.drop(columns=string_cols, inplace=True)

# Step 2: Drop kolom yang >50% NaN (kolom protokol-spesifik)
null_pct = df.isnull().mean()
drop_null = [c for c in null_pct[null_pct > 0.5].index
             if c != 'Attack_type']
print(f"Drop {len(drop_null)} kolom >50% NaN")
df.drop(columns=drop_null, inplace=True)

# Step 3: Fill sisa NaN dengan 0
# (multi-protocol dataset — kolom kosong = protokol lain)
n_nan = df.isnull().sum().sum()
df.fillna(0, inplace=True)
print(f"Filled {n_nan} NaN dengan 0")

# Step 4: Replace Inf
df.replace([np.inf, -np.inf], 0, inplace=True)

# Step 5: Tambah Attack_group
df['Attack_group'] = df['Attack_type'].apply(map_to_group)
df = df[~df['Attack_group'].str.startswith('UNMAPPED_')].copy()

# Acak
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nFinal shape: {df.shape}")
print("\nDistribusi:")
print(df['Attack_group'].value_counts())


# ── CELL 6: Encode ───────────────────────────────────────────
feature_cols = [c for c in df.columns
                if c not in ['Attack_type','Attack_group']]
for col in feature_cols:
    if df[col].dtype == 'object':
        print(f"Encoding: {col}")
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
print(f"Features: {len(feature_cols)} ✓")


# ── CELL 7: Save ─────────────────────────────────────────────
print("\n=== FINAL ===")
print(f"Shape : {df.shape}")
print(f"NaN   : {df.isnull().sum().sum()}")
print(f"RAM   : {df.memory_usage(deep=True).sum()/1024**2:.1f} MB")
print(df['Attack_group'].value_counts())
df.to_csv(OUTPUT, index=False)
print(f"\n✅ Saved: {OUTPUT}")
print("SELESAI! Download edge_iiotset_clean.csv dari panel Output.")

Libraries loaded ✓
Mapping ready ✓
Menjalankan Pass 1 & 2 via subprocess...
Benign:24000
DoS_DDoS:12000
Information_Gathering:6000
MITM:1200
Injection:6000
Malware:12000
Written:61200
✅ 31.2 MB

Membaca /kaggle/working/temp_selected.csv dengan polars...
Shape: (61200, 63)
RAM  : 38.1 MB

Drop 2 kolom string: ['ip.src_host', 'tcp.srcport']
Drop 3 kolom >50% NaN
Filled 51432 NaN dengan 0

Final shape: (61200, 59)

Distribusi:
Attack_group
Benign                   24000
DoS_DDoS                 12000
Malware                  12000
Information_Gathering     6000
Injection                 6000
MITM                      1200
Name: count, dtype: int64
Features: 57 ✓

=== FINAL ===
Shape : (61200, 59)
NaN   : 0
RAM   : 33.3 MB
Attack_group
Benign                   24000
DoS_DDoS                 12000
Malware                  12000
Information_Gathering     6000
Injection                 6000
MITM                      1200
Name: count, dtype: int64

✅ Saved: /kaggle/working/edge_iiotset_clean.c